# 04 · TorchDistributor — M×N GPU

학습 함수는 패키지 [`recommender_pkg.torch_distributor_trainer`](https://github.com/Aiden-Jeon/databricks-distributed-training/blob/main/03-custom-package-script-based/custom_packages/src/recommender_pkg/torch_distributor_trainer.py)의 `train_fn(...)`입니다. 본 노트북은 `TorchDistributor(...).run(...)`로 child 프로세스를 띄워 학습합니다.

02-script-based와 동일한 학습 함수·시그니처를 사용하며, 차이는 import 경로(`recommender_pkg.torch_distributor_trainer`)와 sys.path 보강이 불필요하다는 점뿐입니다.

1×1은 [`02-launch_torch_distributor_1x1.ipynb`](02-launch_torch_distributor_1x1.ipynb), 1×N은 [`03-launch_torch_distributor_1xN.ipynb`](03-launch_torch_distributor_1xN.ipynb)을 참고하세요.

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML, Driver `g5.12xlarge` + Workers `g5.12xlarge` × M. **Autoscaling OFF 필수**이며 Single node 토글도 OFF로 둡니다. Serverless GPU는 multi-node를 지원하지 않습니다.

In [ ]:
%run ./00-setup

## 학습 함수 import 확인 (cloudpickle 회피용 wrapper)

In [ ]:
import os
import sys

# 02-script-based와 달리 SCRIPT_DIR sys.path 보강은 불필요. 패키지가 설치되어 있다.
print(f"recommender_pkg installed:", end=" ")
import recommender_pkg

print(recommender_pkg.__version__)

In [ ]:
def td_train_fn(**kwargs):
    """TorchDistributor child에 보낼 thin wrapper. inline 함수로 by-value pickling.

    패키지가 wheel로 설치되어 있으므로 child의 fresh Python에서도 `recommender_pkg`를
    바로 import할 수 있다. 02-script-based의 sys.path 보강은 불필요.
    """
    from recommender_pkg.torch_distributor_trainer import train_fn

    return train_fn(**kwargs)

## M×N GPU

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2  # M (worker 노드 수)
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="td-MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    td_train_fn,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "td-MxN.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="MxN",
    script_dir="",
)